# Decoder-Only Transformer Training and Testing

In [1]:
import torch
from Transformer import DecoderOnlyTransformer
torch.manual_seed(1337)

if torch.xpu.is_available():
    device = torch.device("xpu")
    print(f"XPU detected: {torch.xpu.get_device_properties(0).name}")
else:
    device = torch.device("cpu")
    print("XPU not available. Using CPU.")

XPU detected: Intel(R) Arc(TM) A770 Graphics


In [2]:
# hyperparameters
batch_size = 32 
block_size = 256  
max_iters = 5000 
eval_interval = 100
learning_rate = 3e-4
eval_iters = 200
n_embd = 384 
n_head = 6  
n_layer = 6  
dropout = 0.2 

In [ ]:
with open('tiny_shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello world"))
print(decode(encode("hello world")))


[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [4]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape)
print(data[:100])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [5]:

def split_data(data, split=0.9):
    n = int(split * len(data))
    return data[:n], data[n:]
train_data, val_data = split_data(data)
print(f"Train data length: {len(train_data)}")
print(f"Val data length: {len(val_data)}")

Train data length: 1003854
Val data length: 111540


In [6]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print(x.shape)
print(y.shape)

torch.Size([32, 256])
torch.Size([32, 256])


In [ ]:
model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    embed_dim=n_embd,  
    n_head=n_head,       
    n_layer=n_layer,    
    max_seq_len=block_size,
    dropout=dropout,   
).to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12000, eta_min=1e-5)

print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

10.788929 M parameters


In [8]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
for iter in range(num_iters):
    
    if iter % eval_interval == 0 or iter == num_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    scheduler.step()



step 0: train loss 1.3450, val loss 1.5848
step 100: train loss 1.3403, val loss 1.5809
step 200: train loss 1.3334, val loss 1.5759
step 300: train loss 1.3301, val loss 1.5690
step 400: train loss 1.3300, val loss 1.5710
step 500: train loss 1.3267, val loss 1.5717
step 600: train loss 1.3226, val loss 1.5698
step 700: train loss 1.3193, val loss 1.5644
step 800: train loss 1.3173, val loss 1.5582
step 900: train loss 1.3147, val loss 1.5634
step 999: train loss 1.3148, val loss 1.5635


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=2000, block_size=block_size)[0].tolist()))


His hight will is not more to did his swith
But wardy of ears, by the husband't.

SICOMINIUS:
How is fubbout signs, he as a generion
To dies no pain?

Shepherd:
Come then retemplation her Happil'd so to
cannot his dumbler'd us you gone; fow, or thus are sun,
Our dilight well have writh nor father writtle every tongues
This body anothory left rame but state,
But usure and amile ine dream, as the came of your cother:
Had not set with us his lose on't pery's shreart
For in that his pails for ath lies: dist we rise;
For that thousand with the Volemongan? capie,
So his with all thus! very way to his vent
Speek the reft of your bosold;
And my dare aved timmetty first
His wordsing this fice, and sign the life heart
You are in on a woman'd to before your heart--blood!
Do save bettlemen, brid: thought a who'st and thus settle reugh
The maids to about seat; and tend them,
To heaver to purish's friards, to another fin!
Have for very heart is a resight,
With vill'd a blask, friendful'd lifes
Will

In [16]:
losses['val']

tensor(1.5635)

In [18]:
save_weights = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'losses': losses,
}
torch.save(save_weights, 'model_weights.pth')
